In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Linear(32 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [ ]:
model = TinyCNN()
x = torch.randn(4, 3, 32, 32)
output = model(x)
print(output.shape)

torch.Size([4, 10])


In [ ]:
class CNNAdapter(nn.Module):
    def __init__(self, channels, bottleneck=4):
        super().__init__()

        self.down_proj = nn.Conv2d(
            channels,
            bottleneck,
            kernel_size=1
        )

        self.up_proj = nn.Conv2d(
            bottleneck,
            channels,
            kernel_size=1
        )

        self.activation = nn.ReLU()

    def forward(self, x):
        residual = x

        x = self.down_proj(x)
        x = self.activation(x)
        x = self.up_proj(x)

        return residual + x

In [ ]:
class CNNWithAdapters(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.adapter1 = CNNAdapter(16)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.adapter2 = CNNAdapter(32)

        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Linear(32 * 8 * 8, num_classes)

    def forward(self, x):

        x = self.conv1(x)
        x = self.adapter1(x)

        x = self.pool(F.relu(x))

        x = self.conv2(x)
        x = self.adapter2(x)

        x = self.pool(F.relu(x))

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [ ]:
model = CNNWithAdapters()

for name, param in model.named_parameters():

    if "adapter" not in name:
        param.requires_grad = False

In [ ]:
total = 0
trainable = 0

for p in model.parameters():

    total += p.numel()

    if p.requires_grad:
        trainable += p.numel()

print("Total params:", total)
print("Trainable params:", trainable)
print("Trainable %:", 100 * trainable / total)

Total params: 26018
Trainable params: 440
Trainable %: 1.6911369052194634


In [ ]:
x = torch.randn(8, 3, 32, 32)
target = torch.randint(0, 10, (8,))

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

for step in range(5):

    optimizer.zero_grad()

    logits = model(x)

    loss = F.cross_entropy(logits, target)

    loss.backward()

    optimizer.step()

    print(f"Step {step} | Loss {loss.item():.4f}")

Step 0 | Loss 2.3053
Step 1 | Loss 2.3022
Step 2 | Loss 2.2991
Step 3 | Loss 2.2961
Step 4 | Loss 2.2930
